# Reelwright in Colab: render a film for free

**Runtime → Run all.** Free Colab is enough (no GPU needed). Drive stores files; Colab does the rendering.

- Your project goes in `MyDrive/reelwright/projects/<name>/` (the bundled ones, like `example`, need nothing).
- The mp4, `.srt`, `.vtt`, manifest and `qa.txt` come back to `MyDrive/reelwright/projects/<name>/out/`.
- Downloads are kept in `MyDrive/reelwright/cache/`, so the second session starts much faster.
- **Disconnected?** Colab drops idle sessions and caps session length. Just **Run all** again: finished work is in
  Drive, so the take, voiceover and (for pvk films) rendered frames are reused. HyperFrames films re-render their frames.
- While the repo is private: add a read-only GitHub token as a Colab secret named `GITHUB_TOKEN` (🔑 in the left bar,
  allow notebook access). Create it at github.com → Settings → Developer settings → Fine-grained tokens, with
  *Contents: read* on this repo only.

In [ ]:
# ── settings ──
PROJECT = "example"          # folder name under projects/ (yours in MyDrive/reelwright/projects/, or a bundled one)
ARGS = ""                    # extra build options, e.g. "--style editorial", "--lang es", "--formats 16x9,9x16"
REPO_URL = "https://github.com/Nexace/reelwright.git"
BRANCH = "main"

In [ ]:
# ── Drive (outside Colab: the folder in $REELWRIGHT_DRIVE, default ./drive) ──
import os, subprocess, sys, base64
try:
    from google.colab import drive as gdrive
    gdrive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/reelwright"
except ImportError:
    DRIVE = os.path.abspath(os.environ.get("REELWRIGHT_DRIVE", "drive"))
os.makedirs(os.path.join(DRIVE, "projects"), exist_ok=True)
print("Drive folder:", DRIVE)

In [ ]:
# ── code: clone (or update) the repo on the VM's local disk; $REELWRIGHT_CODE uses an existing checkout ──
CODE = os.environ.get("REELWRIGHT_CODE") or "/content/reelwright"
auth = []
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")   # sent as a header, never stored in the clone or printed
    auth = ["-c", "http.extraHeader=Authorization: Basic " + base64.b64encode(f"x-access-token:{tok}".encode()).decode()]
except Exception:
    pass                                  # public repo, or not in Colab
git = lambda *a: subprocess.run(["git", *auth, *a], capture_output=True, text=True)
if os.environ.get("REELWRIGHT_CODE"):
    print("using", CODE)
elif os.path.isdir(os.path.join(CODE, ".git")):
    r = git("-C", CODE, "pull", "-q", "--ff-only"); print(r.stderr.strip() or f"updated {CODE}")
else:
    r = git("clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, CODE)
    if r.returncode:
        raise SystemExit("clone failed: " + r.stderr.strip().splitlines()[-1] + "\n(private repo? add the GITHUB_TOKEN secret, see the top)")
    print("cloned into", CODE)
sys.path.insert(0, os.path.join(CODE, "cloud"))
import colab_build as cb

In [ ]:
# ── tools: Node 22, Python libs, Chromium (cached in Drive), then a one-frame smoke test ──
cb.tools(DRIVE)

In [ ]:
# ── build: every step of `reelwright all`, saved to Drive after each; rerun this cell to resume ──
code = cb.build(DRIVE, PROJECT, ARGS.split())